# 00. Configuration

In [1]:
!pip install wandb -q

import warnings
warnings.filterwarnings('ignore')

In [2]:
# ============================================================
# CUHK-X MULTIMODAL HUMAN ACTIVITY RECOGNITION
# Small Model Track
# ============================================================

from pathlib import Path

# -----------------------------
# Dataset
# -----------------------------

BASE_DATA_ROOT = Path(
    "/kaggle/input/datasets/samasiayushman/small-model-track/"
    "Training/Training/data"
)

# Modalities
SKELETON_ROOT = BASE_DATA_ROOT / "Skeleton"
IMU_ROOT = BASE_DATA_ROOT / "IMU"
IR_ROOT = BASE_DATA_ROOT / "IR"
DEPTH_ROOT = BASE_DATA_ROOT / "Depth_Color"
RADAR_ROOT = BASE_DATA_ROOT / "Radar"
THERMAL_ROOT = BASE_DATA_ROOT / "Thermal"


# -----------------------------
# Competition setup
# -----------------------------

NUM_CLASSES = 40

TRAIN_USERS = {
    "user1", "user2", "user3", "user4",
    "user5", "user6", "user7", "user8",
    "user9",

    "user16", "user17", "user18", "user19",
    "user20", "user21", "user22", "user23",
    "user24",
}

# Our fixed local validation subjects
VAL_USERS = {
    "user8",
    "user9",
    "user23",
    "user24",
}

TRAIN_USERS_LOCAL = TRAIN_USERS - VAL_USERS


# Official competition test subjects
TEST_USERS = {
    "user10",
    "user11",
    "user25",
    "user26",
}


# -----------------------------
# Sequence configuration
# -----------------------------

SEQUENCE_LENGTH = 64

# Visual modalities
VISUAL_HEIGHT = 112
VISUAL_WIDTH = 112


# -----------------------------
# Training
# -----------------------------

BATCH_SIZE = 32
NUM_WORKERS = 2

SEED = 42

print("Base data root:", BASE_DATA_ROOT)
print("Number of classes:", NUM_CLASSES)

print("\nTraining users:", len(TRAIN_USERS))
print("Local validation users:", sorted(VAL_USERS))
print("Official test users:", sorted(TEST_USERS))

print("\nDataset roots:")
for name, path in {
    "Skeleton": SKELETON_ROOT,
    "IMU": IMU_ROOT,
    "IR": IR_ROOT,
    "Depth_Color": DEPTH_ROOT,
    "Radar": RADAR_ROOT,
    "Thermal": THERMAL_ROOT,
}.items():
    print(f"{name:12s} -> {path}")

Base data root: /kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data
Number of classes: 40

Training users: 18
Local validation users: ['user23', 'user24', 'user8', 'user9']
Official test users: ['user10', 'user11', 'user25', 'user26']

Dataset roots:
Skeleton     -> /kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Skeleton
IMU          -> /kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/IMU
IR           -> /kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/IR
Depth_Color  -> /kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Depth_Color
Radar        -> /kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar
Thermal      -> /kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Thermal


#  01. Imports & Reproducibility

In [3]:
# ============================================================
# IMPORTS & REPRODUCIBILITY
# ============================================================

import os
import json
import random
import math
import time
import gc

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt


# -----------------------------
# Reproducibility
# -----------------------------

def seed_everything(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Reproducible behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)


# -----------------------------
# Device
# -----------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
Device: cuda
GPU: Tesla T4


## i. Wandb Setup

In [4]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

# Fetch the secret token safely
user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

# Log into WandB
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: hariswarsamasi (hariswarsamasi-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# 02. Dataset Discovery

In [5]:
# ============================================================
# DATASET DISCOVERY
# ============================================================

def get_action_dirs(root):

    return sorted([
        p for p in root.iterdir()
        if p.is_dir()
    ])


action_dirs = get_action_dirs(SKELETON_ROOT)

print("Number of action directories:", len(action_dirs))

print("\nFirst 10 actions:")
for p in action_dirs[:10]:
    print(p.name)

Number of action directories: 40

First 10 actions:
0_Wash_face
10_Stir_drinks
11_Peel_fruits
12_Sweep_the_floor
13_Mop_the_floor
14_Wipe_bowls
15_Wipe_windows_and_tables
16_Fold_clothes
17_Tap_the_keyboard
18_Write


In [6]:
# ============================================================
# BUILD COMMON TRIAL INDEX
# ============================================================

records = []

for action_dir in action_dirs:

    action = action_dir.name

    for user_dir in sorted(action_dir.iterdir()):

        if not user_dir.is_dir():
            continue

        user = user_dir.name

        for trial_dir in sorted(user_dir.iterdir()):

            if not trial_dir.is_dir():
                continue

            trial = trial_dir.name

            records.append({
                "action": action,
                "user": user,
                "trial": trial,
            })


trial_df = pd.DataFrame(records)

print("Total Skeleton trials:", len(trial_df))
print("Actions:", trial_df["action"].nunique())
print("Users:", trial_df["user"].nunique())

print("\nUsers:")
print(sorted(trial_df["user"].unique()))

print("\nExample:")
display(trial_df.head())

Total Skeleton trials: 2931
Actions: 40
Users: 18

Users:
['user1', 'user16', 'user17', 'user18', 'user19', 'user2', 'user20', 'user21', 'user22', 'user23', 'user24', 'user3', 'user4', 'user5', 'user6', 'user7', 'user8', 'user9']

Example:


,action,user,trial
0,0_Wash_face,user16,1-1-1
1,0_Wash_face,user16,1-1-2
2,0_Wash_face,user16,1-1-3
3,0_Wash_face,user18,7-1-1
4,0_Wash_face,user18,7-1-2


# 03. Common Trial Index

In [7]:
def check_modality_trial(root, action, user, trial):
    """
    Check whether a modality contains this exact
    action / user / trial directory.
    """
    trial_dir = root / action / user / trial
    return trial_dir.exists()


modality_roots = {
    "skeleton": SKELETON_ROOT,
    "imu": IMU_ROOT,
    "ir": IR_ROOT,
    "depth": DEPTH_ROOT,
    "radar": RADAR_ROOT,
    "thermal": THERMAL_ROOT,
}


multimodal_df = trial_df.copy()

for modality, root in modality_roots.items():

    multimodal_df[modality] = [
        check_modality_trial(
            root,
            row["action"],
            row["user"],
            row["trial"]
        )
        for _, row in multimodal_df.iterrows()
    ]


print("Total trials:", len(multimodal_df))

print("\nModality availability:")
for modality in modality_roots:
    count = multimodal_df[modality].sum()
    percentage = 100 * count / len(multimodal_df)

    print(
        f"{modality:10s}: "
        f"{count:4d} / {len(multimodal_df)} "
        f"({percentage:.2f}%)"
    )

Total trials: 2931

Modality availability:
skeleton  : 2931 / 2931 (100.00%)
imu       : 2903 / 2931 (99.04%)
ir        : 2931 / 2931 (100.00%)
depth     : 2931 / 2931 (100.00%)
radar     : 2914 / 2931 (99.42%)
thermal   : 2786 / 2931 (95.05%)


## A. SHOW TRIAL AVAILABILITY


In [8]:
display(multimodal_df.head(10))

,action,user,trial,skeleton,imu,ir,depth,radar,thermal
0,0_Wash_face,user16,1-1-1,True,True,True,True,True,True
1,0_Wash_face,user16,1-1-2,True,True,True,True,True,True
2,0_Wash_face,user16,1-1-3,True,True,True,True,True,True
3,0_Wash_face,user18,7-1-1,True,True,True,True,True,True
4,0_Wash_face,user18,7-1-2,True,True,True,True,True,True
5,0_Wash_face,user18,7-1-3,True,True,True,True,True,True
6,0_Wash_face,user20,4-2-1,True,True,True,True,True,True
7,0_Wash_face,user20,4-2-2,True,True,True,True,True,True
8,0_Wash_face,user20,4-2-3,True,True,True,True,True,True
9,0_Wash_face,user21,1-1-1,True,True,True,True,True,True


## B. MISSING MODALITY COUNTS


In [9]:
for modality in modality_roots:

    missing = (~multimodal_df[modality]).sum()

    print(
        f"{modality:10s} missing trials: {missing}"
    )

skeleton   missing trials: 0
imu        missing trials: 28
ir         missing trials: 0
depth      missing trials: 0
radar      missing trials: 17
thermal    missing trials: 145


## C. FIXED SUBJECT-BASED TRAIN / VALIDATION SPLIT


In [10]:
local_train_df = multimodal_df[
    multimodal_df["user"].isin(TRAIN_USERS_LOCAL)
].reset_index(drop=True)

local_val_df = multimodal_df[
    multimodal_df["user"].isin(VAL_USERS)
].reset_index(drop=True)


print("LOCAL TRAIN")
print("Trials:", len(local_train_df))
print("Users:", sorted(local_train_df["user"].unique()))

print("\nLOCAL VALIDATION")
print("Trials:", len(local_val_df))
print("Users:", sorted(local_val_df["user"].unique()))

LOCAL TRAIN
Trials: 2295
Users: ['user1', 'user16', 'user17', 'user18', 'user19', 'user2', 'user20', 'user21', 'user22', 'user3', 'user4', 'user5', 'user6', 'user7']

LOCAL VALIDATION
Trials: 636
Users: ['user23', 'user24', 'user8', 'user9']


## D. LEAKAGE CHECK


In [11]:
train_subjects = set(local_train_df["user"])
val_subjects = set(local_val_df["user"])

overlap = train_subjects & val_subjects

print("Train subjects:", sorted(train_subjects))
print("Val subjects:", sorted(val_subjects))
print("Subject overlap:", overlap)

assert len(overlap) == 0, "SUBJECT LEAKAGE DETECTED!"

Train subjects: ['user1', 'user16', 'user17', 'user18', 'user19', 'user2', 'user20', 'user21', 'user22', 'user3', 'user4', 'user5', 'user6', 'user7']
Val subjects: ['user23', 'user24', 'user8', 'user9']
Subject overlap: set()


## E. CLASS DISTRIBUTION


In [12]:
train_counts = local_train_df["action"].value_counts().sort_index()
val_counts = local_val_df["action"].value_counts().sort_index()

class_distribution = pd.DataFrame({
    "train": train_counts,
    "validation": val_counts
})

display(class_distribution)

print(
    "\nTrain min/max:",
    train_counts.min(),
    train_counts.max()
)

print(
    "Val min/max:",
    val_counts.min(),
    val_counts.max()
)

,train,validation
action,,
0_Wash_face,29,15.0
10_Stir_drinks,98,19.0
11_Peel_fruits,91,15.0
12_Sweep_the_floor,51,12.0
13_Mop_the_floor,48,8.0
14_Wipe_bowls,32,3.0
15_Wipe_windows_and_tables,40,12.0
16_Fold_clothes,15,9.0
17_Tap_the_keyboard,64,22.0



Train min/max: 6 256
Val min/max: 3 79


# 04. Train / Validation Split

## F0: SKELETON + IMU AVAILABILITY


In [13]:

f0_df = multimodal_df[
    multimodal_df["skeleton"] &
    multimodal_df["imu"]
].copy().reset_index(drop=True)

f0_train_df = f0_df[
    f0_df["user"].isin(TRAIN_USERS_LOCAL)
].reset_index(drop=True)

f0_val_df = f0_df[
    f0_df["user"].isin(VAL_USERS)
].reset_index(drop=True)

print("F0 — Skeleton + IMU")
print("=" * 50)

print(f"Total common trials: {len(f0_df)}")
print(f"Train trials:        {len(f0_train_df)}")
print(f"Validation trials:   {len(f0_val_df)}")

print("\nTrain users:")
print(sorted(f0_train_df["user"].unique()))

print("\nValidation users:")
print(sorted(f0_val_df["user"].unique()))

F0 — Skeleton + IMU
Total common trials: 2903
Train trials:        2267
Validation trials:   636

Train users:
['user1', 'user16', 'user17', 'user18', 'user19', 'user2', 'user20', 'user21', 'user22', 'user3', 'user4', 'user5', 'user6', 'user7']

Validation users:
['user23', 'user24', 'user8', 'user9']


In [14]:
# ============================================================
# F0 CLASS DISTRIBUTION
# ============================================================

f0_train_counts = f0_train_df["action"].value_counts().sort_index()
f0_val_counts = f0_val_df["action"].value_counts().sort_index()

f0_distribution = pd.DataFrame({
    "train": f0_train_counts,
    "validation": f0_val_counts
})

display(f0_distribution)

print("\nTrain min/max:",
      f0_train_counts.min(),
      f0_train_counts.max())

print("Val min/max:",
      f0_val_counts.min(),
      f0_val_counts.max())

,train,validation
action,,
0_Wash_face,29,15.0
10_Stir_drinks,96,19.0
11_Peel_fruits,90,15.0
12_Sweep_the_floor,51,12.0
13_Mop_the_floor,45,8.0
14_Wipe_bowls,32,3.0
15_Wipe_windows_and_tables,37,12.0
16_Fold_clothes,15,9.0
17_Tap_the_keyboard,64,22.0



Train min/max: 6 251
Val min/max: 3 79


# 05. Skeleton Dataset

In [15]:
SKELETON_SEQUENCE_LENGTH = 64
SKELETON_NUM_JOINTS = 17
SKELETON_FEATURES_PER_JOINT = 12

# COCO-17 parent structure
SKELETON_PARENTS = [
    -1, 0, 0, 1, 2,
    11, 12,
    5, 6,
    7, 8,
    -1, -1,
    11, 12,
    13, 14
]

In [16]:
def load_skeleton_json(json_path):
    """
    Load skeleton JSON.

    Expected structure:
    [
        {
            "keypoints": [
                [x, y, z],
                ...
            ]
        }
    ]
    """
    with open(json_path, "r") as f:
        data = json.load(f)

    if not data:
        raise ValueError(f"Empty skeleton JSON: {json_path}")

    keypoints = np.asarray(data[0]["keypoints"], dtype=np.float32)

    if keypoints.shape != (17, 3):
        raise ValueError(
            f"Unexpected skeleton shape {keypoints.shape} "
            f"in {json_path}"
        )

    return keypoints

In [17]:
def normalize_skeleton(skeleton):
    """
    Pelvis-center + body-scale normalization.

    Input:
        (17, 3)

    Output:
        (17, 3)
    """

    skeleton = skeleton.astype(np.float32).copy()

    # COCO-17:
    # left hip  = 11
    # right hip = 12
    pelvis = (skeleton[11] + skeleton[12]) / 2.0

    # left shoulder  = 5
    # right shoulder = 6
    shoulder_center = (
        (skeleton[5] + skeleton[6]) / 2.0
    )

    # Move pelvis to origin
    skeleton -= pelvis

    # Body scale
    scale = np.linalg.norm(shoulder_center - pelvis)
    scale = max(scale, 1e-6)

    skeleton /= scale

    return skeleton


def compute_bone_vectors(skeleton):
    """
    Compute bone vectors relative to parent joints.

    Input:
        (17, 3)

    Output:
        (17, 3)
    """

    bones = np.zeros_like(skeleton)

    for joint_idx, parent_idx in enumerate(SKELETON_PARENTS):
        if parent_idx >= 0:
            bones[joint_idx] = (
                skeleton[joint_idx] - skeleton[parent_idx]
            )

    return bones

In [18]:
def temporal_resample(sequence, target_length=64):
    """
    Resample a temporal sequence to target_length.

    Input:
        (T, ...)
    Output:
        (target_length, ...)
    """

    sequence = np.asarray(sequence, dtype=np.float32)

    T = sequence.shape[0]

    if T == target_length:
        return sequence

    if T == 1:
        return np.repeat(
            sequence,
            target_length,
            axis=0
        )

    old_indices = np.linspace(
        0,
        T - 1,
        T
    )

    new_indices = np.linspace(
        0,
        T - 1,
        target_length
    )

    # Flatten everything except time
    flat = sequence.reshape(T, -1)

    resampled = np.empty(
        (target_length, flat.shape[1]),
        dtype=np.float32
    )

    for i in range(flat.shape[1]):
        resampled[:, i] = np.interp(
            new_indices,
            old_indices,
            flat[:, i]
        )

    return resampled.reshape(
        target_length,
        *sequence.shape[1:]
    )

In [19]:
def extract_skeleton_features(trial_dir):
    """
    Extract the 12 features per joint used by the
    previous Skeleton experiments.

    Features:
        1. normalized XYZ       -> 3
        2. velocity             -> 3
        3. acceleration         -> 3
        4. bone vectors         -> 3

    Total = 12 features/joint.

    Output:
        (64, 17, 12)
    """

    predictions_dir = trial_dir / "predictions"

    json_files = sorted(
        predictions_dir.glob("*.json")
    )

    if len(json_files) == 0:
        raise FileNotFoundError(
            f"No skeleton JSON files found in {predictions_dir}"
        )

    skeleton_sequence = []

    for json_path in json_files:
        skeleton = load_skeleton_json(json_path)
        skeleton = normalize_skeleton(skeleton)
        skeleton_sequence.append(skeleton)

    skeleton_sequence = np.stack(
        skeleton_sequence,
        axis=0
    )  # (T, 17, 3)

    # Velocity
    velocity = np.diff(
        skeleton_sequence,
        axis=0,
        prepend=skeleton_sequence[0:1]
    )

    # Acceleration
    acceleration = np.diff(
        velocity,
        axis=0,
        prepend=velocity[0:1]
    )

    # Bone vectors
    bone_vectors = np.stack(
        [
            compute_bone_vectors(frame)
            for frame in skeleton_sequence
        ],
        axis=0
    )

    # Concatenate:
    # XYZ + velocity + acceleration + bones
    features = np.concatenate(
        [
            skeleton_sequence,
            velocity,
            acceleration,
            bone_vectors
        ],
        axis=-1
    )

    # (T, 17, 12)
    features = temporal_resample(
        features,
        SKELETON_SEQUENCE_LENGTH
    )

    return features.astype(np.float32)

In [20]:
class SkeletonDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        trial_dir = (
            SKELETON_ROOT
            / row["action"]
            / row["user"]
            / row["trial"]
        )

        features = extract_skeleton_features(
            trial_dir
        )

        label = int(row["action"].split("_")[0])

        return (
            torch.from_numpy(features),
            torch.tensor(label, dtype=torch.long)
        )

## F. — Skeleton Dataset sanity check


In [21]:
skeleton_train_dataset = SkeletonDataset(f0_train_df)
skeleton_val_dataset = SkeletonDataset(f0_val_df)

x, y = skeleton_train_dataset[0]

print("Skeleton sample shape:", x.shape)
print("Skeleton label:", y.item())
print("dtype:", x.dtype)
print("min:", x.min().item())
print("max:", x.max().item())
print("mean:", x.mean().item())
print("std:", x.std().item())

Skeleton sample shape: torch.Size([64, 17, 12])
Skeleton label: 0
dtype: torch.float32
min: -1.1928156614303589
max: 1.44435453414917
mean: -0.019015803933143616
std: 0.2734016180038452


## Z. Skeleton DataLoaders


In [22]:
skeleton_train_loader = DataLoader(
    skeleton_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

skeleton_val_loader = DataLoader(
    skeleton_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Skeleton train batches:", len(skeleton_train_loader))
print("Skeleton val batches:", len(skeleton_val_loader))

x_batch, y_batch = next(iter(skeleton_train_loader))

print("X batch shape:", x_batch.shape)
print("Y batch shape:", y_batch.shape)
print("X dtype:", x_batch.dtype)
print("Y dtype:", y_batch.dtype)
print("X device:", x_batch.device)
print("Labels:", y_batch[:10].tolist())

Skeleton train batches: 71
Skeleton val batches: 20
X batch shape: torch.Size([32, 64, 17, 12])
Y batch shape: torch.Size([32])
X dtype: torch.float32
Y dtype: torch.int64
X device: cpu
Labels: [6, 13, 4, 36, 12, 32, 34, 37, 2, 9]


In [23]:
# ============================================================
# CELL 13 — S6 Skeleton Encoder
# ============================================================

class S6SkeletonEncoder(nn.Module):

    def __init__(
        self,
        input_size=204,
        projection_size=128,
        hidden_size=128,
        num_layers=2,
        num_heads=4,
        dropout=0.3
    ):
        super().__init__()

        # 17 joints × 12 features = 204
        self.input_projection = nn.Sequential(
            nn.Linear(input_size, projection_size),
            nn.LayerNorm(projection_size),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.lstm = nn.LSTM(
            input_size=projection_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        # BiLSTM output = 128 × 2 = 256
        feature_size = hidden_size * 2

        self.temporal_attention = nn.MultiheadAttention(
            embed_dim=feature_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm = nn.LayerNorm(feature_size)

        # Learned temporal/frame attention
        self.frame_attention = nn.Sequential(
            nn.Linear(feature_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        # This produces the representation used for fusion
        self.embedding_size = feature_size

    def encode(self, x):
        """
        Input:
            [B, 64, 17, 12]

        Output:
            [B, 256]
        """

        B, T, J, F = x.shape

        # Flatten joints/features
        x = x.reshape(B, T, J * F)

        # [B, 64, 204] -> [B, 64, 128]
        x = self.input_projection(x)

        # -> [B, 64, 256]
        x, _ = self.lstm(x)

        # Temporal self-attention
        attended, _ = self.temporal_attention(
            x, x, x
        )

        # Residual connection + normalization
        x = self.norm(x + attended)

        # Learn importance of each frame
        scores = self.frame_attention(x)

        weights = torch.softmax(
            scores,
            dim=1
        )

        # Weighted temporal pooling
        x = torch.sum(
            x * weights,
            dim=1
        )

        return x

    def forward(self, x):
        return self.encode(x)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

skeleton_encoder = S6SkeletonEncoder().to(device)

x_batch, y_batch = next(iter(skeleton_train_loader))

x_batch = x_batch.to(device)

with torch.no_grad():
    skeleton_embedding = skeleton_encoder(x_batch)

print("Input shape:", x_batch.shape)
print("Embedding shape:", skeleton_embedding.shape)
print("Embedding dtype:", skeleton_embedding.dtype)

Input shape: torch.Size([32, 64, 17, 12])
Embedding shape: torch.Size([32, 256])
Embedding dtype: torch.float32


# 06. IMU Dataset

In [24]:
# ============================================================
# CELL 15 — I4 IMU Configuration
# ============================================================

IMU_SEQUENCE_LENGTH = 64

IMU_SENSOR_ORDER = [
    "WTLA",   # Left Arm
    "WTRA",   # Right Arm
    "WTC",    # Chest
    "WTLL",   # Left Leg
    "WTRL"    # Right Leg
]

IMU_FEATURE_COLUMNS = [
    "加速度X(g)",
    "加速度Y(g)",
    "加速度Z(g)",
    "角速度X(°/s)",
    "角速度Y(°/s)",
    "角速度Z(°/s)"
]

IMU_INPUT_SIZE = len(IMU_SENSOR_ORDER) * len(IMU_FEATURE_COLUMNS)

print("Sensors:", IMU_SENSOR_ORDER)
print("Features per sensor:", len(IMU_FEATURE_COLUMNS))
print("Total features per timestep:", IMU_INPUT_SIZE)

Sensors: ['WTLA', 'WTRA', 'WTC', 'WTLL', 'WTRL']
Features per sensor: 6
Total features per timestep: 30


## B. Load and synchronize one IMU trial

In [25]:
def load_imu_trial(trial_dir):
    """
    Load the five IMU sensors from one trial.

    Returns:
        {
            sensor_name: DataFrame
        }
    """

    up_file = trial_dir / "up(LA+RA+C).csv"
    down_file = trial_dir / "down(LL+RL).csv"

    if not up_file.exists() or not down_file.exists():
        raise FileNotFoundError(
            f"Missing IMU files in {trial_dir}"
        )

    up_df = pd.read_csv(up_file)
    down_df = pd.read_csv(down_file)

    df = pd.concat(
        [up_df, down_df],
        ignore_index=True
    )

    # Parse timestamps
    df["时间"] = pd.to_datetime(
        df["时间"],
        errors="coerce"
    )

    # Remove invalid rows
    df = df.dropna(
        subset=["时间", "设备名称"]
    )

    sensor_data = {}

    for sensor in IMU_SENSOR_ORDER:

        sensor_df = df[
            df["设备名称"].astype(str).str.startswith(sensor)
        ].copy()

        sensor_df = sensor_df.sort_values(
            "时间"
        )

        sensor_df = sensor_df[
            ["时间"] + IMU_FEATURE_COLUMNS
        ].copy()

        sensor_df[IMU_FEATURE_COLUMNS] = (
            sensor_df[IMU_FEATURE_COLUMNS]
            .apply(pd.to_numeric, errors="coerce")
        )

        sensor_df = sensor_df.dropna(
            subset=IMU_FEATURE_COLUMNS
        )

        sensor_data[sensor] = sensor_df

    return sensor_data

## C. IMU synchronization

In [26]:
def synchronize_imu(
    sensor_data,
    sequence_length=64
):
    """
    Synchronize all five IMU sensors onto a common
    temporal grid and resample to 64 frames.

    Output:
        (64, 30)
    """

    # Find the common temporal overlap
    start_time = max(
        df["时间"].min()
        for df in sensor_data.values()
        if len(df) > 0
    )

    end_time = min(
        df["时间"].max()
        for df in sensor_data.values()
        if len(df) > 0
    )

    if end_time <= start_time:
        raise ValueError(
            "No common temporal overlap between IMU sensors."
        )

    # Common timeline
    target_times = pd.date_range(
        start=start_time,
        end=end_time,
        periods=sequence_length
    )

    target_seconds = (
        target_times.astype("int64") / 1e9
    ).to_numpy()

    all_sensor_features = []

    for sensor in IMU_SENSOR_ORDER:

        df = sensor_data[sensor]

        if len(df) < 2:
            raise ValueError(
                f"Sensor {sensor} has insufficient data."
            )

        source_seconds = (
            df["时间"].astype("int64") / 1e9
        ).to_numpy()

        sensor_features = []

        for feature in IMU_FEATURE_COLUMNS:

            values = (
                df[feature]
                .astype(np.float32)
                .to_numpy()
            )

            interpolated = np.interp(
                target_seconds,
                source_seconds,
                values
            )

            sensor_features.append(
                interpolated
            )

        # (64, 6)
        sensor_features = np.stack(
            sensor_features,
            axis=1
        )

        all_sensor_features.append(
            sensor_features
        )

    # (64, 5, 6)
    synchronized = np.stack(
        all_sensor_features,
        axis=1
    )

    # Flatten sensors
    # (64, 5, 6) -> (64, 30)
    synchronized = synchronized.reshape(
        sequence_length,
        -1
    )

    return synchronized.astype(np.float32)

In [27]:
# ============================================================
# CELL 18 — IMU synchronization sanity check
# ============================================================

sample_row = f0_train_df.iloc[0]

sample_imu_trial = (
    IMU_ROOT
    / sample_row["action"]
    / sample_row["user"]
    / sample_row["trial"]
)

sensor_data = load_imu_trial(
    sample_imu_trial
)

print("Trial:", sample_imu_trial)
print()

for sensor in IMU_SENSOR_ORDER:
    print(
        sensor,
        "rows:",
        len(sensor_data[sensor])
    )

imu_sample = synchronize_imu(
    sensor_data,
    IMU_SEQUENCE_LENGTH
)

print()
print("IMU sample shape:", imu_sample.shape)
print("dtype:", imu_sample.dtype)
print("min:", np.min(imu_sample))
print("max:", np.max(imu_sample))
print("mean:", np.mean(imu_sample))
print("std:", np.std(imu_sample))

Trial: /kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/IMU/0_Wash_face/user16/1-1-1

WTLA rows: 45
WTRA rows: 46
WTC rows: 46
WTLL rows: 44
WTRL rows: 46

IMU sample shape: (64, 30)
dtype: float32
min: -108.17394
max: 92.83788
mean: 0.39140373
std: 10.282346


## D. IMU training normalization statistics

In [28]:
print("Computing IMU normalization statistics...")
print("Training trials:", len(f0_train_df))

# We calculate statistics per sensor × feature.
# Shape: (5 sensors, 6 features)

imu_sum = np.zeros(
    (len(IMU_SENSOR_ORDER), len(IMU_FEATURE_COLUMNS)),
    dtype=np.float64
)

imu_sum_sq = np.zeros_like(imu_sum)
imu_count = np.zeros_like(imu_sum, dtype=np.int64)

for idx, row in f0_train_df.iterrows():

    trial_dir = (
        IMU_ROOT
        / row["action"]
        / row["user"]
        / row["trial"]
    )

    try:
        sensor_data = load_imu_trial(trial_dir)
        sequence = synchronize_imu(
            sensor_data,
            IMU_SEQUENCE_LENGTH
        )

        # (64, 30) -> (64, 5, 6)
        sequence = sequence.reshape(
            IMU_SEQUENCE_LENGTH,
            len(IMU_SENSOR_ORDER),
            len(IMU_FEATURE_COLUMNS)
        )

        valid = np.isfinite(sequence)

        imu_sum += np.where(
            valid,
            sequence,
            0
        ).sum(axis=0)

        imu_sum_sq += np.where(
            valid,
            sequence ** 2,
            0
        ).sum(axis=0)

        imu_count += valid.sum(axis=0)

    except Exception as e:
        continue

    if (idx + 1) % 500 == 0:
        print(
            f"Processed {idx + 1}/{len(f0_train_df)}"
        )

imu_mean = imu_sum / np.maximum(imu_count, 1)

imu_variance = (
    imu_sum_sq / np.maximum(imu_count, 1)
    - imu_mean ** 2
)

imu_variance = np.maximum(
    imu_variance,
    1e-8
)

imu_std = np.sqrt(imu_variance)

print("\nNormalization statistics computed.")
print("Mean shape:", imu_mean.shape)
print("Std shape:", imu_std.shape)

print("\nMean:")
print(imu_mean)

print("\nStd:")
print(imu_std)

Computing IMU normalization statistics...
Training trials: 2267
Processed 500/2267
Processed 1000/2267
Processed 1500/2267
Processed 2000/2267

Normalization statistics computed.
Mean shape: (5, 6)
Std shape: (5, 6)

Mean:
[[-0.34742488  0.20479452  0.26163209 -0.29040349 -1.67002889 -0.98411671]
 [ 0.33568027  0.20783301  0.32133636  0.90558618  2.67798786  2.71813569]
 [-0.04481415  0.79413201 -0.10782608 -1.28675364  1.48912781 -0.13205434]
 [-0.05770191  0.78932931 -0.05539985 -0.28900713  0.70100786 -0.11024557]
 [ 0.05343975  0.84168749 -0.08283601 -0.60211366 -0.00606721 -0.66025915]]

Std:
[[  0.60299489   0.71772565   0.53411586  69.30133874 103.23974317
   86.82104813]
 [  0.61559026   0.71588514   0.52497034  71.68974573 104.10524373
   86.58482081]
 [  0.16835922   0.49538102   0.34679285  28.14689793  37.38931142
   15.24167692]
 [  0.28100786   0.57683786   0.32819254  50.51455883  50.18931344
   27.41363781]
 [  0.32276691   0.46834463   0.38813124  50.36168425  57.77715

In [29]:
# ============================================================
# CELL 23 — Keep only valid IMU trials
# ============================================================

def is_valid_imu_trial(row):
    trial_dir = (
        IMU_ROOT
        / row["action"]
        / row["user"]
        / row["trial"]
    )

    try:
        sensor_data = load_imu_trial(trial_dir)

        # Every sensor must have enough samples
        for sensor in IMU_SENSOR_ORDER:
            if len(sensor_data[sensor]) < 2:
                return False

        # Synchronization must have a common overlap
        _ = synchronize_imu(
            sensor_data,
            IMU_SEQUENCE_LENGTH
        )

        return True

    except Exception:
        return False


print("Checking IMU training trials...")

train_valid_mask = []

for i, (_, row) in enumerate(
    f0_train_df.iterrows()
):

    valid = is_valid_imu_trial(row)
    train_valid_mask.append(valid)

    if (i + 1) % 500 == 0:
        print(
            f"Checked {i + 1}/{len(f0_train_df)}"
        )

valid_imu_train_df = f0_train_df[
    train_valid_mask
].reset_index(drop=True)


print()
print("Original IMU train trials:",
      len(f0_train_df))

print("Valid IMU train trials:",
      len(valid_imu_train_df))

print("Removed IMU train trials:",
      len(f0_train_df) - len(valid_imu_train_df))
# ============================================================
# CELL 24 — Keep only valid IMU validation trials
# ============================================================

print("Checking IMU validation trials...")

val_valid_mask = []

for i, (_, row) in enumerate(
    f0_val_df.iterrows()
):

    valid = is_valid_imu_trial(row)
    val_valid_mask.append(valid)

    if (i + 1) % 200 == 0:
        print(
            f"Checked {i + 1}/{len(f0_val_df)}"
        )

valid_imu_val_df = f0_val_df[
    val_valid_mask
].reset_index(drop=True)


print()
print("Original IMU val trials:",
      len(f0_val_df))

print("Valid IMU val trials:",
      len(valid_imu_val_df))

print("Removed IMU val trials:",
      len(f0_val_df) - len(valid_imu_val_df))

Checking IMU training trials...
Checked 500/2267
Checked 1000/2267
Checked 1500/2267
Checked 2000/2267

Original IMU train trials: 2267
Valid IMU train trials: 2108
Removed IMU train trials: 159
Checking IMU validation trials...
Checked 200/636
Checked 400/636
Checked 600/636

Original IMU val trials: 636
Valid IMU val trials: 628
Removed IMU val trials: 8


## E. I4 Normalized IMU Dataset

In [30]:
class I4IMUDataset(Dataset):

    def __init__(
        self,
        dataframe,
        imu_mean,
        imu_std
    ):
        self.df = dataframe.reset_index(drop=True)

        # Store normalization statistics
        # Shape: (5, 6)
        self.mean = imu_mean.astype(np.float32)
        self.std = imu_std.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        trial_dir = (
            IMU_ROOT
            / row["action"]
            / row["user"]
            / row["trial"]
        )

        try:

            sensor_data = load_imu_trial(
                trial_dir
            )

            sequence = synchronize_imu(
                sensor_data,
                IMU_SEQUENCE_LENGTH
            )

        except Exception as e:

            raise RuntimeError(
                f"Failed to load IMU trial:\n"
                f"{trial_dir}\n"
                f"Error: {e}"
            )

        # ----------------------------------------------------
        # Reshape:
        # (64, 30) -> (64, 5, 6)
        # ----------------------------------------------------

        sequence = sequence.reshape(
            IMU_SEQUENCE_LENGTH,
            len(IMU_SENSOR_ORDER),
            len(IMU_FEATURE_COLUMNS)
        )

        # ----------------------------------------------------
        # Sensor-wise normalization
        # ----------------------------------------------------

        sequence = (
            sequence - self.mean[None, :, :]
        ) / (
            self.std[None, :, :] + 1e-6
        )

        # ----------------------------------------------------
        # Flatten:
        # (64, 5, 6) -> (64, 30)
        # ----------------------------------------------------

        sequence = sequence.reshape(
            IMU_SEQUENCE_LENGTH,
            IMU_INPUT_SIZE
        )

        # Safety check
        if not np.isfinite(sequence).all():
            raise RuntimeError(
                f"NaN/Inf detected in IMU trial:\n"
                f"{trial_dir}"
            )

        label = int(
            row["action"].split("_")[0]
        )

        return (
            torch.from_numpy(
                sequence.astype(np.float32)
            ),
            torch.tensor(
                label,
                dtype=torch.long
            )
        )

imu_train_dataset = I4IMUDataset(
    valid_imu_train_df,
    imu_mean,
    imu_std
)

imu_val_dataset = I4IMUDataset(
    valid_imu_val_df,
    imu_mean,
    imu_std
)

print(
    "IMU train samples:",
    len(imu_train_dataset)
)

print(
    "IMU val samples:",
    len(imu_val_dataset)
)

IMU train samples: 2108
IMU val samples: 628


## F. CHECK

In [31]:
imu_x, imu_y = imu_train_dataset[0]

print("IMU sample shape:", imu_x.shape)
print("IMU label:", imu_y.item())
print("dtype:", imu_x.dtype)

print("min:", imu_x.min().item())
print("max:", imu_x.max().item())
print("mean:", imu_x.mean().item())
print("std:", imu_x.std().item())

print(
    "Contains NaN:",
    torch.isnan(imu_x).any().item()
)

print(
    "Contains Inf:",
    torch.isinf(imu_x).any().item()
)

IMU sample shape: torch.Size([64, 30])
IMU label: 0
dtype: torch.float32
min: -1.9180785417556763
max: 0.9806889891624451
mean: -0.049317918717861176
std: 0.5002427697181702
Contains NaN: False
Contains Inf: False


In [32]:
# ============================================================
# CELL 26 — I4 IMU DataLoaders
# ============================================================

imu_train_loader = DataLoader(
    imu_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

imu_val_loader = DataLoader(
    imu_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(
    "IMU train batches:",
    len(imu_train_loader)
)

print(
    "IMU val batches:",
    len(imu_val_loader)
)

IMU train batches: 66
IMU val batches: 20


In [33]:
# ============================================================
# CELL 28 — I4 IMU Encoder
# ============================================================

class I4IMUEncoder(nn.Module):

    def __init__(
        self,
        input_size=30,
        projection_size=128,
        hidden_size=128,
        num_layers=2,
        num_heads=4,
        dropout=0.3
    ):
        super().__init__()

        self.input_projection = nn.Sequential(
            nn.Linear(input_size, projection_size),
            nn.LayerNorm(projection_size),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.lstm = nn.LSTM(
            input_size=projection_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        # BiLSTM:
        # 128 hidden × 2 directions = 256
        feature_size = hidden_size * 2

        self.temporal_attention = nn.MultiheadAttention(
            embed_dim=feature_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm = nn.LayerNorm(feature_size)

        self.frame_attention = nn.Sequential(
            nn.Linear(feature_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.embedding_size = feature_size

    def encode(self, x):
        """
        Input:
            [B, 64, 30]

        Output:
            [B, 256]
        """

        # [B, 64, 30]
        x = self.input_projection(x)

        # [B, 64, 256]
        x, _ = self.lstm(x)

        # Temporal self-attention
        attended, _ = self.temporal_attention(
            x, x, x
        )

        # Residual + LayerNorm
        x = self.norm(
            x + attended
        )

        # Learned frame importance
        scores = self.frame_attention(x)

        weights = torch.softmax(
            scores,
            dim=1
        )

        # Weighted temporal pooling
        x = torch.sum(
            x * weights,
            dim=1
        )

        return x

    def forward(self, x):
        return self.encode(x)

imu_encoder = I4IMUEncoder().to(device)

imu_x_batch, imu_y_batch = next(
    iter(imu_train_loader)
)

imu_x_batch = imu_x_batch.to(device)

with torch.no_grad():
    imu_embedding = imu_encoder(
        imu_x_batch
    )

print("Input shape:", imu_x_batch.shape)
print("Embedding shape:", imu_embedding.shape)
print("Embedding dtype:", imu_embedding.dtype)

Input shape: torch.Size([32, 64, 30])
Embedding shape: torch.Size([32, 256])
Embedding dtype: torch.float32


# 07. IR Dataset

# 08. Depth_Color Dataset

# 09. Skeleton Encoder

## 0. Individual modality classifier


In [34]:
class ModalityClassifier(nn.Module):

    def __init__(
        self,
        encoder,
        embedding_size=256,
        num_classes=40
    ):
        super().__init__()

        self.encoder = encoder

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(embedding_size, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        embedding = self.encoder.encode(x)

        logits = self.classifier(
            embedding
        )

        return logits


skeleton_model = ModalityClassifier(
    encoder=S6SkeletonEncoder(),
    embedding_size=256,
    num_classes=NUM_CLASSES
).to(device)

imu_model = ModalityClassifier(
    encoder=I4IMUEncoder(),
    embedding_size=256,
    num_classes=NUM_CLASSES
).to(device)

print("Skeleton parameters:",sum(p.numel() for p in skeleton_model.parameters()))

print("IMU parameters:",sum(p.numel() for p in imu_model.parameters()))

# ============================================================
# CELL 33 — Checkpoint directory
# ============================================================

CHECKPOINT_DIR = Path(
    "/kaggle/working/cuhk_x_checkpoints"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Checkpoint directory:",
    CHECKPOINT_DIR
)

Skeleton parameters: 1020713
IMU parameters: 998441
Checkpoint directory: /kaggle/working/cuhk_x_checkpoints


In [35]:
def train_modality_model(
    model,
    train_loader,
    val_loader,
    run_name,
    checkpoint_path,
    epochs=30,
    learning_rate=1e-3,
    weight_decay=1e-4,
):
    """
    Train one modality model.

    Features:
      - CrossEntropyLoss
      - AdamW optimizer
      - W&B logging
      - best validation checkpoint
      - validation accuracy
      - learning-rate logging
    """

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    # Reduce LR when validation accuracy stops improving
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=3
    )

    wandb.init(
        project="CIUX",
        name=run_name,
        config={
            "model": run_name,
            "epochs": epochs,
            "batch_size": BATCH_SIZE,
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "num_classes": NUM_CLASSES,
            "device": str(device)
        },
        reinit="finish_previous"
    )

    best_val_accuracy = -1.0
    best_epoch = -1

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
        "learning_rate": []
    }

    for epoch in range(1, epochs + 1):
        # TRAIN
        
        model.train()

        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        train_start = time.time()

        for x, y in train_loader:

            x = x.to(
                device,
                non_blocking=True
            )

            y = y.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(x)

            loss = criterion(
                logits,
                y
            )

            loss.backward()

            # Prevent occasional exploding gradients
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            train_loss_sum += (
                loss.item() * y.size(0)
            )

            predictions = logits.argmax(
                dim=1
            )

            train_correct += (
                predictions == y
            ).sum().item()

            train_total += y.size(0)

        train_loss = (
            train_loss_sum / train_total
        )

        train_accuracy = (
            train_correct / train_total
        )
        

        # VALIDATION
        model.eval()

        val_loss_sum = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for x, y in val_loader:

                x = x.to(
                    device,
                    non_blocking=True
                )

                y = y.to(
                    device,
                    non_blocking=True
                )

                logits = model(x)

                loss = criterion(
                    logits,
                    y
                )

                val_loss_sum += (
                    loss.item() * y.size(0)
                )

                predictions = logits.argmax(
                    dim=1
                )

                val_correct += (
                    predictions == y
                ).sum().item()

                val_total += y.size(0)

        val_loss = (
            val_loss_sum / val_total
        )

        val_accuracy = (
            val_correct / val_total
        )


        
        # SCHEDULER
        # ====================================================

        scheduler.step(
            val_accuracy
        )

        current_lr = optimizer.param_groups[0]["lr"]

        epoch_time = (
            time.time() - train_start
        )

        # ====================================================
        # SAVE HISTORY
        # ====================================================

        history["train_loss"].append(
            train_loss
        )

        history["train_accuracy"].append(
            train_accuracy
        )

        history["val_loss"].append(
            val_loss
        )

        history["val_accuracy"].append(
            val_accuracy
        )

        history["learning_rate"].append(
            current_lr
        )


        # BEST CHECKPOINT
        # ====================================================

        is_best = (
            val_accuracy > best_val_accuracy
        )

        if is_best:

            best_val_accuracy = val_accuracy
            best_epoch = epoch

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_accuracy": best_val_accuracy,
                    "run_name": run_name
                },
                checkpoint_path
            )


        
        # W&B
        # ====================================================

        wandb.log(
            {
                "epoch": epoch,

                "train/loss": train_loss,
                "train/accuracy": train_accuracy,

                "val/loss": val_loss,
                "val/accuracy": val_accuracy,

                "learning_rate": current_lr,

                "best/val_accuracy": best_val_accuracy,
                "best/epoch": best_epoch,

                "epoch_time_sec": epoch_time
            }
        )


        
        # PRINT
        # ====================================================

        marker = " ★ BEST" if is_best else ""

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy*100:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy*100:.2f}% | "
            f"LR: {current_lr:.2e} | "
            f"Time: {epoch_time:.1f}s"
            f"{marker}"
        )


    # FINISH W&B
    # ========================================================

    wandb.summary["best_val_accuracy"] = (
        best_val_accuracy
    )

    wandb.summary["best_epoch"] = (
        best_epoch
    )

    wandb.finish()

    print()
    print("=" * 70)
    print(f"Training complete: {run_name}")
    print(
        f"Best validation accuracy: "
        f"{best_val_accuracy*100:.2f}%"
    )
    print(
        f"Best epoch: {best_epoch}"
    )
    print(
        f"Checkpoint: {checkpoint_path}"
    )
    print("=" * 70)

    return history

In [36]:
skeleton_history = train_modality_model(
    model=skeleton_model,
    train_loader=skeleton_train_loader,
    val_loader=skeleton_val_loader,

    run_name="skeleton_s6",

    checkpoint_path=(
        CHECKPOINT_DIR
        / "skeleton_s6_best.pt"
    ),

    epochs=30,
    learning_rate=1e-3,
    weight_decay=1e-4
)

wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260911_072155-l1qjq1sf
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run skeleton_s6
wandb: ⭐️ View project at https://wandb.ai/hariswarsamasi-indian-institute-of-technology-madras/CIUX
wandb: 🚀 View run at https://wandb.ai/hariswarsamasi-indian-institute-of-technology-madras/CIUX/runs/l1qjq1sf


Epoch 01/30 | Train Loss: 3.0140 | Train Acc: 21.26% | Val Loss: 2.8041 | Val Acc: 24.21% | LR: 1.00e-03 | Time: 204.7s ★ BEST
Epoch 02/30 | Train Loss: 2.3281 | Train Acc: 32.64% | Val Loss: 2.4614 | Val Acc: 29.40% | LR: 1.00e-03 | Time: 38.4s ★ BEST
Epoch 03/30 | Train Loss: 2.0068 | Train Acc: 39.66% | Val Loss: 2.3924 | Val Acc: 33.96% | LR: 1.00e-03 | Time: 34.2s ★ BEST
Epoch 04/30 | Train Loss: 1.7960 | Train Acc: 45.96% | Val Loss: 2.3115 | Val Acc: 37.74% | LR: 1.00e-03 | Time: 42.7s ★ BEST
Epoch 05/30 | Train Loss: 1.6152 | Train Acc: 51.04% | Val Loss: 2.3774 | Val Acc: 36.16% | LR: 1.00e-03 | Time: 36.5s
Epoch 06/30 | Train Loss: 1.4664 | Train Acc: 54.61% | Val Loss: 2.3403 | Val Acc: 33.02% | LR: 1.00e-03 | Time: 34.0s
Epoch 07/30 | Train Loss: 1.3600 | Train Acc: 57.26% | Val Loss: 2.1171 | Val Acc: 43.40% | LR: 1.00e-03 | Time: 32.3s ★ BEST
Epoch 08/30 | Train Loss: 1.2615 | Train Acc: 61.14% | Val Loss: 2.3353 | Val Acc: 39.94% | LR: 1.00e-03 | Time: 33.3s
Epoch 09/30 

wandb: updating run metadata


Epoch 30/30 | Train Loss: 0.1338 | Train Acc: 95.46% | Val Loss: 3.5018 | Val Acc: 45.75% | LR: 6.25e-05 | Time: 33.2s


wandb: uploading wandb-summary.json; uploading config.yaml; uploading output.log
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:        best/epoch ▁▁▂▂▂▂▃▃▃▃▃▄▅▅▅▅▅▆▆▆▆▆████████
wandb: best/val_accuracy ▁▃▄▅▅▅▇▇▇▇▇▇▇▇▇▇▇█████████████
wandb:             epoch ▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
wandb:    epoch_time_sec █▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▁▁▁▁▁
wandb:     learning_rate ██████████▄▄▄▄▄▄▂▂▂▂▂▁▁▁▁▁▁▁▁▁
wandb:    train/accuracy ▁▂▃▃▄▄▄▅▅▅▅▆▆▇▇▇▇▇▇▇██████████
wandb:        train/loss █▆▆▅▅▄▄▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
wandb:      val/accuracy ▁▃▄▅▅▄▇▆▆▆▆▇▇▇▆▇▇██▇█▇███▇▇█▇▇
wandb:          val/loss ▄▃▂▂▂▂▁▂▂▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇██
wandb: 
wandb: Run summary:
wandb:        best/epoch 24
wandb: best/val_accuracy 0.47642
wandb:        best_epoch 24
wandb: best_val_accuracy 0.47642
wandb:             epoch 30
wandb:    epoch_time_sec 33.15178
wandb:     learning_rate 6e-05
wandb:    train/accuracy 0.95457
wandb:        train/loss 0.1338
wandb:      val/accu


Training complete: skeleton_s6
Best validation accuracy: 47.64%
Best epoch: 24
Checkpoint: /kaggle/working/cuhk_x_checkpoints/skeleton_s6_best.pt


In [37]:
imu_history = train_modality_model(
    model=imu_model,
    train_loader=imu_train_loader,
    val_loader=imu_val_loader,
    run_name="imu_i4",
    checkpoint_path=CHECKPOINT_DIR / "imu_i4_best.pt",
    epochs=30,
    learning_rate=1e-3,
    weight_decay=1e-4
)

wandb: setting up run 3krw28cl
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260911_074309-3krw28cl
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run imu_i4
wandb: ⭐️ View project at https://wandb.ai/hariswarsamasi-indian-institute-of-technology-madras/CIUX
wandb: 🚀 View run at https://wandb.ai/hariswarsamasi-indian-institute-of-technology-madras/CIUX/runs/3krw28cl


Epoch 01/30 | Train Loss: 3.1481 | Train Acc: 18.50% | Val Loss: 3.1281 | Val Acc: 17.20% | LR: 1.00e-03 | Time: 94.2s ★ BEST
Epoch 02/30 | Train Loss: 2.5155 | Train Acc: 30.60% | Val Loss: 3.0305 | Val Acc: 21.18% | LR: 1.00e-03 | Time: 89.8s ★ BEST
Epoch 03/30 | Train Loss: 2.0450 | Train Acc: 40.32% | Val Loss: 3.1735 | Val Acc: 22.45% | LR: 1.00e-03 | Time: 87.7s ★ BEST
Epoch 04/30 | Train Loss: 1.7535 | Train Acc: 47.58% | Val Loss: 3.1772 | Val Acc: 23.25% | LR: 1.00e-03 | Time: 87.6s ★ BEST
Epoch 05/30 | Train Loss: 1.5011 | Train Acc: 51.76% | Val Loss: 3.3720 | Val Acc: 23.57% | LR: 1.00e-03 | Time: 88.0s ★ BEST
Epoch 06/30 | Train Loss: 1.2734 | Train Acc: 60.20% | Val Loss: 3.4982 | Val Acc: 24.84% | LR: 1.00e-03 | Time: 87.9s ★ BEST
Epoch 07/30 | Train Loss: 1.1190 | Train Acc: 64.61% | Val Loss: 3.5459 | Val Acc: 24.20% | LR: 1.00e-03 | Time: 87.8s
Epoch 08/30 | Train Loss: 0.9236 | Train Acc: 71.11% | Val Loss: 3.8013 | Val Acc: 25.00% | LR: 1.00e-03 | Time: 87.9s ★ BEST

wandb: updating run metadata


Epoch 30/30 | Train Loss: 0.0406 | Train Acc: 98.39% | Val Loss: 6.5910 | Val Acc: 27.23% | LR: 6.25e-05 | Time: 88.5s


wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:        best/epoch ▁▂▂▃▃▄▄▅▆▆▆▆██████████████████
wandb: best/val_accuracy ▁▃▄▄▅▅▅▆▆▇▇▇██████████████████
wandb:             epoch ▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
wandb:    epoch_time_sec █▃▁▁▁▁▁▁▂▂▃▂▃▄▅▄▅▃▃▃▃▃▂▂▁▂▂▂▁▂
wandb:     learning_rate ████████████████▄▄▄▄▂▂▂▂▁▁▁▁▁▁
wandb:    train/accuracy ▁▂▃▄▄▅▅▆▆▆▇▇▇▇▇▇▇█████████████
wandb:        train/loss █▇▆▅▄▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:      val/accuracy ▁▃▄▄▅▅▅▆▆▇▆▆█▆█▆▆▇▅▇▆▇▆▆▆▆▆▆▆▇
wandb:          val/loss ▁▁▁▁▂▂▂▂▃▃▄▄▅▅▅▅▅▆▇▆▇▇▇█▇█████
wandb: 
wandb: Run summary:
wandb:        best/epoch 13
wandb: best/val_accuracy 0.29299
wandb:        best_epoch 13
wandb: best_val_accuracy 0.29299
wandb:             epoch 30
wandb:    epoch_time_sec 88.5121
wandb:     learning_rate 6e-05
wandb:    train/accuracy 0.98387
wandb:        train/loss 0.04064
wandb:      val/accu


Training complete: imu_i4
Best validation accuracy: 29.30%
Best epoch: 13
Checkpoint: /kaggle/working/cuhk_x_checkpoints/imu_i4_best.pt


# 13. Fusion Architecture

In [38]:
# Load best Skeleton checkpoint
skeleton_ckpt = torch.load(
    CHECKPOINT_DIR / "skeleton_s6_best.pt",
    map_location=device
)

skeleton_model.load_state_dict(
    skeleton_ckpt["model_state_dict"]
)

print(
    "Skeleton best epoch:",
    skeleton_ckpt["epoch"],
    "Val Acc:",
    f'{skeleton_ckpt["best_val_accuracy"]:.2%}'
)


# Load best IMU checkpoint
imu_ckpt = torch.load(
    CHECKPOINT_DIR / "imu_i4_best.pt",
    map_location=device
)

imu_model.load_state_dict(
    imu_ckpt["model_state_dict"]
)

print(
    "IMU best epoch:",
    imu_ckpt["epoch"],
    "Val Acc:",
    f'{imu_ckpt["best_val_accuracy"]:.2%}'
)

Skeleton best epoch: 24 Val Acc: 47.64%
IMU best epoch: 13 Val Acc: 29.30%


In [39]:
def load_skeleton_trial(trial_dir, sequence_length=SEQUENCE_LENGTH):
    """
    Load one Skeleton trial.

    Skeleton JSON files are stored inside:
        <trial>/predictions/

    Returns:
        np.ndarray of shape [64, 17, 12]
    """

    predictions_dir = trial_dir / "predictions"

    json_files = list(
        predictions_dir.glob("*.json")
    )

    if len(json_files) == 0:
        raise ValueError(
            f"No JSON skeleton files found: {predictions_dir}"
        )

    # Sort by filename. The filenames contain frame indices.
    json_files = sorted(
        json_files,
        key=lambda p: p.name
    )

    frames = []

    for json_file in json_files:

        with open(json_file, "r") as f:
            data = json.load(f)

        # -------------------------------------------------
        # Each JSON contains a list with keypoints
        # -------------------------------------------------
        if isinstance(data, list):
            if len(data) == 0:
                continue

            keypoints = data[0]["keypoints"]

        else:
            keypoints = data["keypoints"]

        keypoints = np.asarray(
            keypoints,
            dtype=np.float32
        )

        # [17, 3]
        keypoints = keypoints.reshape(17, 3)

        frames.append(keypoints)

    if len(frames) == 0:
        raise ValueError(
            f"No valid skeleton frames found: {predictions_dir}"
        )

    skeleton = np.stack(
        frames,
        axis=0
    )

    # =====================================================
    # Pelvis-centered normalization
    # =====================================================

    pelvis = (
        skeleton[:, 11, :] +
        skeleton[:, 12, :]
    ) / 2.0

    skeleton = (
        skeleton -
        pelvis[:, None, :]
    )

    # =====================================================
    # Scale normalization
    # =====================================================

    shoulder_center = (
        skeleton[:, 5, :] +
        skeleton[:, 6, :]
    ) / 2.0

    scale = np.linalg.norm(
        shoulder_center,
        axis=1,
        keepdims=True
    )

    scale = np.clip(
        scale,
        1e-6,
        None
    )

    skeleton = (
        skeleton /
        scale[:, None, :]
    )

    # =====================================================
    # Velocity
    # =====================================================

    velocity = np.diff(
        skeleton,
        axis=0,
        prepend=skeleton[0:1]
    )

    # =====================================================
    # Acceleration
    # =====================================================

    acceleration = np.diff(
        velocity,
        axis=0,
        prepend=velocity[0:1]
    )

    # =====================================================
    # Bone vectors
    # =====================================================

    parents = [
        -1, 0, 0, 1, 2,
        11, 12,
        5, 6,
        7, 8,
        -1, -1,
        11, 12,
        13, 14
    ]

    bone_vectors = np.zeros_like(
        skeleton
    )

    for joint, parent in enumerate(parents):

        if parent >= 0:

            bone_vectors[:, joint, :] = (
                skeleton[:, joint, :]
                - skeleton[:, parent, :]
            )

    # =====================================================
    # Combine
    # =====================================================

    features = np.concatenate(
        [
            skeleton,       # 3
            velocity,       # 3
            acceleration,   # 3
            bone_vectors    # 3
        ],
        axis=-1
    )

    # [T, 17, 12]
    assert features.shape[1:] == (
        17, 12
    )

    # =====================================================
    # Temporal resampling
    # =====================================================

    T = features.shape[0]

    if T != sequence_length:

        old_indices = np.linspace(
            0,
            T - 1,
            T
        )

        new_indices = np.linspace(
            0,
            T - 1,
            sequence_length
        )

        resampled = np.zeros(
            (
                sequence_length,
                17,
                12
            ),
            dtype=np.float32
        )

        for joint in range(17):

            for feature in range(12):

                resampled[:, joint, feature] = np.interp(
                    new_indices,
                    old_indices,
                    features[:, joint, feature]
                )

        features = resampled

    return features.astype(
        np.float32
    )
    
f0_train_paired_df = valid_imu_train_df.copy()
f0_val_paired_df = valid_imu_val_df.copy()

print("F0 paired train:", len(f0_train_paired_df))
print("F0 paired val:  ", len(f0_val_paired_df))

print(
    "Train users:",
    sorted(f0_train_paired_df["user"].unique())
)

print(
    "Val users:",
    sorted(f0_val_paired_df["user"].unique())
)

F0 paired train: 2108
F0 paired val:   628
Train users: ['user1', 'user16', 'user17', 'user18', 'user19', 'user2', 'user20', 'user21', 'user22', 'user3', 'user4', 'user5', 'user6', 'user7']
Val users: ['user23', 'user24', 'user8', 'user9']


In [40]:
class F0SkeletonIMUFusion(nn.Module):

    def __init__(
        self,
        skeleton_encoder,
        imu_encoder,
        num_classes=40
    ):
        super().__init__()

        self.skeleton_encoder = skeleton_encoder
        self.imu_encoder = imu_encoder

        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.2),

            nn.Linear(128, num_classes)
        )

    def forward(self, skeleton, imu):

        s = self.skeleton_encoder.encode(
            skeleton
        )

        i = self.imu_encoder.encode(
            imu
        )

        fused = torch.cat(
            [s, i],
            dim=1
        )

        return self.classifier(fused)

class SkeletonIMUDataset(Dataset):

    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        action = row["action"]
        user = row["user"]
        trial = row["trial"]

        # =====================================================
        # Skeleton
        # =====================================================
        skeleton_dir = (
            SKELETON_ROOT
            / action
            / user
            / trial
        )

        skeleton = load_skeleton_trial(skeleton_dir)

        skeleton = torch.tensor(
            skeleton,
            dtype=torch.float32
        )

        # Expected:
        # [64, 17, 12]
        assert skeleton.shape == (64, 17, 12), (
            f"Unexpected skeleton shape: {skeleton.shape}"
        )

        # =====================================================
        # IMU
        # =====================================================
        imu_dir = (
            IMU_ROOT
            / action
            / user
            / trial
        )

        sensor_data = load_imu_trial(imu_dir)

        imu = synchronize_imu(
            sensor_data,
            sequence_length=IMU_SEQUENCE_LENGTH
        )

        # [64, 30]
        imu = imu.reshape(
            IMU_SEQUENCE_LENGTH,
            5,
            6
        )

        # Training-set normalization
        imu = (
            imu - IMU_MEAN
        ) / IMU_STD

        imu = imu.reshape(
            IMU_SEQUENCE_LENGTH,
            IMU_INPUT_SIZE
        )

        imu = torch.tensor(
            imu,
            dtype=torch.float32
        )

        assert imu.shape == (64, 30), (
            f"Unexpected IMU shape: {imu.shape}"
        )

        # =====================================================
        # Label
        # =====================================================
        label = int(action.split("_")[0])

        label = torch.tensor(
            label,
            dtype=torch.long
        )

        return skeleton, imu, label

In [41]:
f0_train_dataset = SkeletonIMUDataset(
    f0_train_paired_df
)

f0_val_dataset = SkeletonIMUDataset(
    f0_val_paired_df
)

print("F0 train:", len(f0_train_dataset))
print("F0 val:  ", len(f0_val_dataset))

F0 train: 2108
F0 val:   628


In [42]:
IMU_MEAN = np.array([
    [-0.34742488, 0.20479452, 0.26163209, -0.29040349, -1.67002889, -0.98411671],
    [ 0.33568027, 0.20783301, 0.32133636,  0.90558618,  2.67798786,  2.71813569],
    [-0.04481415, 0.79413201, -0.10782608, -1.28675364,  1.48912781, -0.13205434],
    [-0.05770191, 0.78932931, -0.05539985, -0.28900713,  0.70100786, -0.11024557],
    [ 0.05343975, 0.84168749, -0.08283601, -0.60211366, -0.00606721, -0.66025915]
], dtype=np.float32)

IMU_STD = np.array([
    [0.60299489, 0.71772565, 0.53411586, 69.30133874, 103.23974317, 86.82104813],
    [0.61559026, 0.71588514, 0.52497034, 71.68974573, 104.10524373, 86.58482081],
    [0.16835922, 0.49538102, 0.34679285, 28.14689793, 37.38931142, 15.24167692],
    [0.28100786, 0.57683786, 0.32819254, 50.51455883, 50.18963781, 27.41363781],
    [0.32276691, 0.46834463, 0.38813124, 50.36168425, 57.77715868, 38.84884282]
], dtype=np.float32)

print("IMU_MEAN:", IMU_MEAN.shape)
print("IMU_STD: ", IMU_STD.shape)
print("Minimum STD:", IMU_STD.min())
skeleton_x, imu_x, y = f0_train_dataset[0]

print("Skeleton:", skeleton_x.shape, skeleton_x.dtype)
print("IMU:     ", imu_x.shape, imu_x.dtype)
print("Label:   ", y.item())

print("\nSkeleton:")
print("  min:", skeleton_x.min().item())
print("  max:", skeleton_x.max().item())
print("  mean:", skeleton_x.mean().item())
print("  std:", skeleton_x.std().item())

print("\nIMU:")
print("  min:", imu_x.min().item())
print("  max:", imu_x.max().item())
print("  mean:", imu_x.mean().item())
print("  std:", imu_x.std().item())

print("\nNaN:")
print("  Skeleton:", torch.isnan(skeleton_x).any().item())
print("  IMU:     ", torch.isnan(imu_x).any().item())

print("\nInf:")
print("  Skeleton:", torch.isinf(skeleton_x).any().item())
print("  IMU:     ", torch.isinf(imu_x).any().item())

IMU_MEAN: (5, 6)
IMU_STD:  (5, 6)
Minimum STD: 0.16835922
Skeleton: torch.Size([64, 17, 12]) torch.float32
IMU:      torch.Size([64, 30]) torch.float32
Label:    0

Skeleton:
  min: -1.1928156614303589
  max: 1.44435453414917
  mean: -0.019015805795788765
  std: 0.2734016180038452

IMU:
  min: -1.9180822372436523
  max: 0.9806889891624451
  mean: -0.0493178628385067
  std: 0.5002437829971313

NaN:
  Skeleton: False
  IMU:      False

Inf:
  Skeleton: False
  IMU:      False


In [43]:
f0_train_loader = DataLoader(
    f0_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

f0_val_loader = DataLoader(
    f0_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("F0 train batches:", len(f0_train_loader))
print("F0 val batches:  ", len(f0_val_loader))

F0 train batches: 66
F0 val batches:   20


In [44]:
skeleton_batch, imu_batch, labels = next(
    iter(f0_train_loader)
)

print("Skeleton batch:", skeleton_batch.shape)
print("IMU batch:     ", imu_batch.shape)
print("Labels:        ", labels.shape)

print("Skeleton dtype:", skeleton_batch.dtype)
print("IMU dtype:     ", imu_batch.dtype)
print("Labels dtype:  ", labels.dtype)

Skeleton batch: torch.Size([32, 64, 17, 12])
IMU batch:      torch.Size([32, 64, 30])
Labels:         torch.Size([32])
Skeleton dtype: torch.float32
IMU dtype:      torch.float32
Labels dtype:   torch.int64


In [45]:
class F0SkeletonIMUFusion(nn.Module):

    def __init__(
        self,
        skeleton_encoder,
        imu_encoder,
        num_classes=40,
        dropout=0.3
    ):
        super().__init__()

        self.skeleton_encoder = skeleton_encoder
        self.imu_encoder = imu_encoder

        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.2),

            nn.Linear(128, num_classes)
        )

    def forward(self, skeleton, imu):

        skeleton_embedding = (
            self.skeleton_encoder.encode(skeleton)
        )

        imu_embedding = (
            self.imu_encoder.encode(imu)
        )

        fused = torch.cat(
            [
                skeleton_embedding,
                imu_embedding
            ],
            dim=1
        )

        logits = self.classifier(fused)

        return logits

In [46]:
# ---------------------------------------------------------
# Create fresh encoders
# ---------------------------------------------------------

skeleton_encoder = S6SkeletonEncoder().to(device)
imu_encoder = I4IMUEncoder().to(device)


# ---------------------------------------------------------
# Load Skeleton checkpoint
# ---------------------------------------------------------

skeleton_checkpoint = torch.load(
    CHECKPOINT_DIR / "skeleton_s6_best.pt",
    map_location=device
)

skeleton_model_temp = ModalityClassifier(
    skeleton_encoder,
    embedding_size=256,
    num_classes=40
).to(device)

skeleton_model_temp.load_state_dict(
    skeleton_checkpoint["model_state_dict"]
)


# ---------------------------------------------------------
# Load IMU checkpoint
# ---------------------------------------------------------

imu_checkpoint = torch.load(
    CHECKPOINT_DIR / "imu_i4_best.pt",
    map_location=device
)

imu_model_temp = ModalityClassifier(
    imu_encoder,
    embedding_size=256,
    num_classes=40
).to(device)

imu_model_temp.load_state_dict(
    imu_checkpoint["model_state_dict"]
)


print("Skeleton checkpoint loaded.")
print("IMU checkpoint loaded.")

for param in skeleton_encoder.parameters():
    param.requires_grad = False

for param in imu_encoder.parameters():
    param.requires_grad = False

print(
    "Skeleton trainable:",
    sum(p.numel() for p in skeleton_encoder.parameters() if p.requires_grad)
)

print(
    "IMU trainable:",
    sum(p.numel() for p in imu_encoder.parameters() if p.requires_grad)
)

Skeleton checkpoint loaded.
IMU checkpoint loaded.
Skeleton trainable: 0
IMU trainable: 0


In [47]:
f0_model = F0SkeletonIMUFusion(
    skeleton_encoder=skeleton_encoder,
    imu_encoder=imu_encoder,
    num_classes=NUM_CLASSES
).to(device)

trainable_params = sum(
    p.numel()
    for p in f0_model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in f0_model.parameters()
)

print("Total parameters:    ", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
f0_model.eval()

with torch.no_grad():

    skeleton_batch = skeleton_batch.to(device)
    imu_batch = imu_batch.to(device)

    logits = f0_model(
        skeleton_batch,
        imu_batch
    )

print("Skeleton input:", skeleton_batch.shape)
print("IMU input:     ", imu_batch.shape)
print("Output logits: ", logits.shape)

Total parameters:     2,112,938
Trainable parameters: 169,896
Skeleton input: torch.Size([32, 64, 17, 12])
IMU input:      torch.Size([32, 64, 30])
Output logits:  torch.Size([32, 40])


In [48]:
def train_f0_fusion(
    model,
    train_loader,
    val_loader,
    run_name,
    checkpoint_path,
    epochs=10,
    learning_rate=1e-3,
    weight_decay=1e-4,
):

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    # Only train parameters that require gradients
    optimizer = torch.optim.AdamW(
        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2
    )

    wandb.init(
        project="CIUX",
        name=run_name,
        config={
            "architecture": "F0_Skeleton_IMU",
            "epochs": epochs,
            "batch_size": BATCH_SIZE,
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "frozen_encoders": True,
            "skeleton_embedding": 256,
            "imu_embedding": 256,
            "fusion_embedding": 512,
            "num_classes": NUM_CLASSES
        },
        reinit="finish_previous"
    )

    best_val_accuracy = -1.0
    best_epoch = -1

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": []
    }

    for epoch in range(1, epochs + 1):

        start_time = time.time()

        # =====================================================
        # TRAIN
        # =====================================================

        model.train()

        # Keep frozen encoders in eval mode so their Dropout
        # does not introduce noise during fusion-head training.
        model.skeleton_encoder.eval()
        model.imu_encoder.eval()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for skeleton, imu, labels in train_loader:

            skeleton = skeleton.to(
                device,
                non_blocking=True
            )

            imu = imu.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad()

            logits = model(
                skeleton,
                imu
            )

            loss = criterion(
                logits,
                labels
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            train_loss += (
                loss.item() *
                labels.size(0)
            )

            predictions = logits.argmax(dim=1)

            train_correct += (
                predictions == labels
            ).sum().item()

            train_total += labels.size(0)

        train_loss /= train_total

        train_accuracy = (
            train_correct /
            train_total
        )

        # =====================================================
        # VALIDATION
        # =====================================================

        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for skeleton, imu, labels in val_loader:

                skeleton = skeleton.to(
                    device,
                    non_blocking=True
                )

                imu = imu.to(
                    device,
                    non_blocking=True
                )

                labels = labels.to(
                    device,
                    non_blocking=True
                )

                logits = model(
                    skeleton,
                    imu
                )

                loss = criterion(
                    logits,
                    labels
                )

                val_loss += (
                    loss.item() *
                    labels.size(0)
                )

                predictions = logits.argmax(dim=1)

                val_correct += (
                    predictions == labels
                ).sum().item()

                val_total += labels.size(0)

        val_loss /= val_total

        val_accuracy = (
            val_correct /
            val_total
        )

        scheduler.step(
            val_accuracy
        )

        current_lr = optimizer.param_groups[0]["lr"]

        # =====================================================
        # BEST CHECKPOINT
        # =====================================================

        is_best = (
            val_accuracy >
            best_val_accuracy
        )

        if is_best:

            best_val_accuracy = val_accuracy
            best_epoch = epoch

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_accuracy": best_val_accuracy,
                    "run_name": run_name
                },
                checkpoint_path
            )

        # =====================================================
        # HISTORY
        # =====================================================

        history["train_loss"].append(
            train_loss
        )

        history["train_accuracy"].append(
            train_accuracy
        )

        history["val_loss"].append(
            val_loss
        )

        history["val_accuracy"].append(
            val_accuracy
        )

        elapsed = time.time() - start_time

        # =====================================================
        # W&B
        # =====================================================

        wandb.log(
            {
                "epoch": epoch,
                "train/loss": train_loss,
                "train/accuracy": train_accuracy,
                "val/loss": val_loss,
                "val/accuracy": val_accuracy,
                "learning_rate": current_lr,
                "best/val_accuracy": best_val_accuracy,
                "best/epoch": best_epoch,
                "epoch_time_sec": elapsed
            }
        )

        marker = " ★ BEST" if is_best else ""

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.2%} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy:.2%} | "
            f"LR: {current_lr:.2e} | "
            f"Time: {elapsed:.1f}s"
            f"{marker}"
        )

    wandb.summary["best_val_accuracy"] = (
        best_val_accuracy
    )

    wandb.summary["best_epoch"] = (
        best_epoch
    )

    wandb.finish()

    print("\n" + "=" * 60)
    print("F0 TRAINING COMPLETE")
    print("=" * 60)
    print(
        f"Best Val Accuracy: "
        f"{best_val_accuracy:.2%}"
    )
    print(
        f"Best Epoch: {best_epoch}"
    )
    print(
        f"Checkpoint: {checkpoint_path}"
    )

    return history

In [49]:
f0_history = train_f0_fusion(
    model=f0_model,
    train_loader=f0_train_loader,
    val_loader=f0_val_loader,
    run_name="F0_skeleton_imu_frozen",
    checkpoint_path=CHECKPOINT_DIR / "F0_skeleton_imu_best.pt",
    epochs=30,
    learning_rate=1e-3,
    weight_decay=1e-4
)

wandb: setting up run m0xj3eeo
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260911_082749-m0xj3eeo
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run F0_skeleton_imu_frozen
wandb: ⭐️ View project at https://wandb.ai/hariswarsamasi-indian-institute-of-technology-madras/CIUX
wandb: 🚀 View run at https://wandb.ai/hariswarsamasi-indian-institute-of-technology-madras/CIUX/runs/m0xj3eeo


Epoch 01/30 | Train Loss: 1.4389 | Train Acc: 72.58% | Val Loss: 2.0985 | Val Acc: 48.57% | LR: 1.00e-03 | Time: 217.5s ★ BEST
Epoch 02/30 | Train Loss: 0.2336 | Train Acc: 93.83% | Val Loss: 2.1945 | Val Acc: 48.73% | LR: 1.00e-03 | Time: 143.0s ★ BEST
Epoch 03/30 | Train Loss: 0.1345 | Train Acc: 95.97% | Val Loss: 2.4684 | Val Acc: 48.25% | LR: 1.00e-03 | Time: 142.9s
Epoch 04/30 | Train Loss: 0.1049 | Train Acc: 96.73% | Val Loss: 2.6377 | Val Acc: 47.77% | LR: 1.00e-03 | Time: 138.5s
Epoch 05/30 | Train Loss: 0.1042 | Train Acc: 96.73% | Val Loss: 2.7607 | Val Acc: 47.45% | LR: 5.00e-04 | Time: 137.4s
Epoch 06/30 | Train Loss: 0.0762 | Train Acc: 97.39% | Val Loss: 2.7506 | Val Acc: 47.29% | LR: 5.00e-04 | Time: 139.0s
Epoch 07/30 | Train Loss: 0.0689 | Train Acc: 97.58% | Val Loss: 2.8498 | Val Acc: 47.93% | LR: 5.00e-04 | Time: 139.6s
Epoch 08/30 | Train Loss: 0.0703 | Train Acc: 97.53% | Val Loss: 2.8677 | Val Acc: 47.93% | LR: 2.50e-04 | Time: 145.1s
Epoch 09/30 | Train Loss: 

wandb: uploading data; updating run metadata


Epoch 30/30 | Train Loss: 0.0343 | Train Acc: 98.43% | Val Loss: 3.0677 | Val Acc: 47.77% | LR: 1.95e-06 | Time: 144.6s


wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading history steps 29-29, summary, console lines 29-29
wandb: 
wandb: Run history:
wandb:        best/epoch ▁█████████████████████████████
wandb: best/val_accuracy ▁█████████████████████████████
wandb:             epoch ▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
wandb:    epoch_time_sec ▇▁▁▁▁▁▁▂▁▂▂▂▂▂▂▁▂▂▃▂▂▂█▅▆▂▁▂▁▁
wandb:     learning_rate ████▄▄▄▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:    train/accuracy ▁▇▇▇▇█████████████████████████
wandb:        train/loss █▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:      val/accuracy ▇█▆▃▂▁▄▄▃▃▃▅▃▄▆▅▅▅▃▃▅▃▃▃▃▄▄▃▃▃
wandb:          val/loss ▁▂▄▅▆▆▆▇▇▇▇▇██████████████████
wandb: 
wandb: Run summary:
wandb:        best/epoch 2
wandb: best/val_accuracy 0.48726
wandb:        best_epoch 2
wandb: best_val_accuracy 0.48726
wandb:             epoch 30
wandb:    epoch_time_sec 144.55193
wandb:     learning_rate 0.0
wandb:    train/accuracy 0.98435
wandb:        train/loss 0.03429
wandb:      va


F0 TRAINING COMPLETE
Best Val Accuracy: 48.73%
Best Epoch: 2
Checkpoint: /kaggle/working/cuhk_x_checkpoints/F0_skeleton_imu_best.pt
